In [ ]:
import re
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os

outdir = "outputs/archived_graph_figures"
os.makedirs(outdir, exist_ok=True)

raw = r"""
n_samples method epsilon                loss    sparsity                feas         acc           time  n_success  n_attempts  n_fail
       100   BAPG   1e-02 5.97e-01 ± 1.69e-01 0.83 ± 0.14 8.03e-01 ± 1.27e-01 0.01 ± 0.01    0.02 ± 0.01       20.0          20     0.0
       100   BAPG   1e-01 3.10e-02 ± 2.03e-02 0.48 ± 0.14 2.27e-02 ± 8.11e-03 0.12 ± 0.24    0.50 ± 0.17       20.0          20     0.0
       100   BAPG   1e+00 3.05e-02 ± 1.61e-02 0.00 ± 0.00 1.46e-03 ± 4.72e-04 0.25 ± 0.39    0.58 ± 0.08       20.0          20     0.0
       100     CG       - 3.60e-02 ± 2.03e-02 0.99 ± 0.00 0.00e+00 ± 0.00e+00 0.02 ± 0.01    0.00 ± 0.00       20.0          20     0.0
       100    PGD   1e-03 3.11e-02 ± 1.55e-02 0.00 ± 0.00 5.54e-05 ± 2.28e-05 0.21 ± 0.35    0.38 ± 0.43       20.0          20     0.0
       100    PGD   1e-02 5.35e-02 ± 2.23e-02 0.00 ± 0.00 5.85e-17 ± 2.82e-18 0.06 ± 0.03    0.00 ± 0.00       20.0          20     0.0
       100    PGD   1e-01 5.36e-02 ± 2.23e-02 0.00 ± 0.00 5.83e-17 ± 3.88e-18 0.04 ± 0.02    0.00 ± 0.00       20.0          20     0.0
       100    PGD   1e+00 5.36e-02 ± 2.23e-02 0.00 ± 0.00 5.76e-17 ± 3.43e-18 0.04 ± 0.02    0.00 ± 0.00       20.0          20     0.0
       100    PPA   1e-03 3.53e-02 ± 1.99e-02 0.97 ± 0.02 4.69e-05 ± 1.43e-05 0.10 ± 0.11    1.83 ± 1.21       20.0          20     0.0
       100    PPA   1e-02 3.08e-02 ± 1.60e-02 0.43 ± 0.31 1.04e-05 ± 5.46e-06 0.25 ± 0.39    2.61 ± 1.96       20.0          20     0.0
       100    PPA   1e-01 3.07e-02 ± 1.60e-02 0.01 ± 0.01 1.52e-06 ± 9.80e-07 0.25 ± 0.38  15.27 ± 16.51       20.0          20     0.0
       100    PPA   1e+00 3.07e-02 ± 1.60e-02 0.00 ± 0.00 1.92e-07 ± 1.11e-07 0.24 ± 0.38  38.57 ± 11.41       20.0          20     0.0
       100    iPG       - 3.21e-02 ± 1.50e-02 0.99 ± 0.00 7.63e-09 ± 1.17e-08 0.17 ± 0.29    0.92 ± 0.24       20.0          20     0.0
       200   BAPG   1e-02 8.05e-01 ± 1.16e-01 0.96 ± 0.05 7.18e-01 ± 6.87e-02 0.01 ± 0.00    0.21 ± 0.13       20.0          20     0.0
       200   BAPG   1e-01 1.06e-01 ± 1.38e-02 0.23 ± 0.27 4.19e-03 ± 5.84e-03 0.67 ± 0.46    0.91 ± 1.05       20.0          20     0.0
       200   BAPG   1e+00 1.10e-01 ± 1.48e-02 0.00 ± 0.00 5.18e-04 ± 5.35e-04 0.55 ± 0.46    1.64 ± 0.85       20.0          20     0.0
       200     CG       - 1.26e-01 ± 5.86e-04 0.99 ± 0.00 0.00e+00 ± 0.00e+00 0.01 ± 0.01    0.02 ± 0.00       20.0          20     0.0
       200    PGD   1e-03 1.11e-01 ± 1.45e-02 0.00 ± 0.00 3.25e-05 ± 3.02e-05 0.51 ± 0.46    0.49 ± 0.51       20.0          20     0.0
       200    PGD   1e-02 1.43e-01 ± 4.66e-04 0.00 ± 0.00 5.04e-17 ± 1.95e-18 0.03 ± 0.01    0.01 ± 0.00       20.0          20     0.0
       200    PGD   1e-01 1.43e-01 ± 4.65e-04 0.00 ± 0.00 4.82e-17 ± 1.77e-18 0.03 ± 0.02    0.01 ± 0.00       20.0          20     0.0
       200    PGD   1e+00 1.43e-01 ± 4.65e-04 0.00 ± 0.00 5.01e-17 ± 1.70e-18 0.02 ± 0.01    0.01 ± 0.00       20.0          20     0.0
       200    PPA   1e-03 1.16e-01 ± 1.39e-02 0.84 ± 0.31 5.10e-05 ± 3.86e-05 0.39 ± 0.43    3.77 ± 2.80       20.0          20     0.0
       200    PPA   1e-02 1.12e-01 ± 1.49e-02 0.53 ± 0.50 1.50e-05 ± 1.37e-05 0.51 ± 0.46    8.97 ± 8.62       20.0          20     0.0
       200    PPA   1e-01 1.10e-01 ± 1.49e-02 0.02 ± 0.02 2.48e-06 ± 2.41e-06 0.55 ± 0.46  22.54 ± 24.27       20.0          20     0.0
       200    PPA   1e+00 1.10e-01 ± 1.50e-02 0.00 ± 0.00 2.67e-07 ± 2.71e-07 0.55 ± 0.46  37.65 ± 19.73       20.0          20     0.0
       200    iPG       - 1.13e-01 ± 1.45e-02 0.99 ± 0.00 1.34e-09 ± 1.64e-09 0.46 ± 0.46    5.63 ± 2.78       20.0          20     0.0
       300   BAPG   1e-02 1.14e+00 ± 2.08e-01 0.98 ± 0.03 7.48e-01 ± 5.49e-02 0.00 ± 0.00    0.53 ± 0.27       20.0          20     0.0
       300   BAPG   1e-01 6.03e-02 ± 3.41e-02 0.29 ± 0.31 6.15e-03 ± 6.43e-03 0.51 ± 0.50    3.68 ± 3.55       20.0          20     0.0
       300   BAPG   1e+00 6.09e-02 ± 3.45e-02 0.00 ± 0.00 5.35e-04 ± 5.65e-04 0.52 ± 0.49    4.70 ± 2.54       20.0          20     0.0
       300     CG       - 8.34e-02 ± 2.27e-02 1.00 ± 0.00 0.00e+00 ± 0.00e+00 0.01 ± 0.01    0.05 ± 0.01       20.0          20     0.0
       300    PGD   1e-03 6.07e-02 ± 3.43e-02 0.00 ± 0.00 1.85e-05 ± 1.92e-05 0.51 ± 0.50   6.18 ± 24.41       20.0          20     0.0
       300    PGD   1e-02 1.03e-01 ± 1.98e-02 0.00 ± 0.00 4.50e-17 ± 1.59e-18 0.02 ± 0.01    0.01 ± 0.00       20.0          20     0.0
       300    PGD   1e-01 1.03e-01 ± 1.98e-02 0.00 ± 0.00 4.52e-17 ± 1.59e-18 0.02 ± 0.01    0.01 ± 0.00       20.0          20     0.0
       300    PGD   1e+00 1.03e-01 ± 1.98e-02 0.00 ± 0.00 4.44e-17 ± 1.49e-18 0.02 ± 0.01    0.01 ± 0.00       20.0          20     0.0
       300    PPA   1e-03 6.65e-02 ± 3.18e-02 0.87 ± 0.28 3.07e-05 ± 2.50e-05 0.43 ± 0.47    5.39 ± 5.12       20.0          20     0.0
       300    PPA   1e-02 6.10e-02 ± 3.45e-02 0.49 ± 0.50 6.25e-06 ± 6.82e-06 0.52 ± 0.49  13.11 ± 14.38       20.0          20     0.0
       300    PPA   1e-01 6.32e-02 ± 3.35e-02 0.02 ± 0.03 1.09e-06 ± 1.17e-06 0.48 ± 0.48  23.05 ± 24.50       20.0          20     0.0
       300    PPA   1e+00 6.33e-02 ± 3.36e-02 0.00 ± 0.00 1.28e-07 ± 1.22e-07 0.48 ± 0.48  56.78 ± 25.95       20.0          20     0.0
       300    iPG       - 5.85e-02 ± 3.52e-02 1.00 ± 0.00 1.15e-09 ± 7.60e-10 0.57 ± 0.49  15.66 ± 13.86       20.0          20     0.0
       400   BAPG   1e-02 1.16e+00 ± 3.41e-01 0.99 ± 0.01 7.15e-01 ± 1.84e-01 0.00 ± 0.00    1.82 ± 1.52       20.0          20     0.0
       400   BAPG   1e-01 4.40e-02 ± 2.39e-02 0.21 ± 0.27 4.05e-03 ± 5.10e-03 0.62 ± 0.47    5.69 ± 6.49       20.0          20     0.0
       400   BAPG   1e+00 4.41e-02 ± 2.40e-02 0.00 ± 0.00 4.17e-04 ± 5.48e-04 0.65 ± 0.44    8.28 ± 4.43       20.0          20     0.0
       400     CG       - 7.40e-02 ± 4.87e-04 1.00 ± 0.00 0.00e+00 ± 0.00e+00 0.00 ± 0.00    0.12 ± 0.01       20.0          20     0.0
       400    PGD   1e-03 4.44e-02 ± 2.44e-02 0.00 ± 0.00 1.14e-05 ± 1.45e-05 0.62 ± 0.48   8.37 ± 33.05       20.0          20     0.0
       400    PGD   1e-02 9.26e-02 ± 4.58e-04 0.00 ± 0.00 4.66e-17 ± 1.35e-18 0.02 ± 0.01    0.03 ± 0.01       20.0          20     0.0
       400    PGD   1e-01 9.26e-02 ± 4.58e-04 0.00 ± 0.00 4.69e-17 ± 1.76e-18 0.01 ± 0.00    0.03 ± 0.00       20.0          20     0.0
       400    PGD   1e+00 9.26e-02 ± 4.58e-04 0.00 ± 0.00 4.60e-17 ± 1.44e-18 0.01 ± 0.00    0.03 ± 0.00       20.0          20     0.0
       400    PPA   1e-03 4.63e-02 ± 2.43e-02 0.79 ± 0.37 1.40e-04 ± 3.98e-04 0.63 ± 0.44    4.89 ± 5.89       20.0          20     0.0
       400    PPA   1e-02 4.44e-02 ± 2.44e-02 0.39 ± 0.50 4.21e-06 ± 5.16e-06 0.64 ± 0.46  13.13 ± 16.80       20.0          20     0.0
       400    PPA   1e-01 4.43e-02 ± 2.42e-02 0.01 ± 0.02 6.74e-07 ± 8.25e-07 0.64 ± 0.46  39.81 ± 46.61       20.0          20     0.0
       400    PPA   1e+00 4.47e-02 ± 2.45e-02 0.00 ± 0.00 1.17e-07 ± 1.20e-07 0.63 ± 0.46  68.51 ± 25.22       20.0          20     0.0
       400    iPG       - 3.95e-02 ± 2.28e-02 1.00 ± 0.00 1.44e-09 ± 1.16e-09 0.71 ± 0.45  31.89 ± 39.18       20.0          20     0.0
       500   BAPG   1e-02 9.09e-01 ± 4.23e-01 0.99 ± 0.00 5.86e-01 ± 2.37e-01 0.00 ± 0.00    7.45 ± 4.61       20.0          20     0.0
       500   BAPG   1e-01 4.71e-02 ± 2.50e-02 0.20 ± 0.21 3.62e-03 ± 3.85e-03 0.57 ± 0.49  12.33 ± 12.26       20.0          20     0.0
       500   BAPG   1e+00 5.01e-02 ± 2.57e-02 0.00 ± 0.00 4.38e-04 ± 3.78e-04 0.52 ± 0.49   18.35 ± 7.76       20.0          20     0.0
       500     CG       - 7.52e-02 ± 3.04e-04 1.00 ± 0.00 0.00e+00 ± 0.00e+00 0.00 ± 0.00    0.21 ± 0.03       20.0          20     0.0
       500    PGD   1e-03 5.66e-02 ± 2.38e-02 0.00 ± 0.00 1.59e-05 ± 1.20e-05 0.36 ± 0.48    2.24 ± 1.66       20.0          20     0.0
       500    PGD   1e-02 9.19e-02 ± 2.52e-04 0.00 ± 0.00 4.04e-17 ± 1.06e-18 0.01 ± 0.00    0.05 ± 0.01       20.0          20     0.0
       500    PGD   1e-01 9.19e-02 ± 2.52e-04 0.00 ± 0.00 4.22e-17 ± 1.24e-18 0.01 ± 0.00    0.05 ± 0.01       20.0          20     0.0
       500    PGD   1e+00 9.19e-02 ± 2.52e-04 0.00 ± 0.00 4.08e-17 ± 1.06e-18 0.01 ± 0.00    0.05 ± 0.01       20.0          20     0.0
       500    PPA   1e-03 4.52e-02 ± 2.53e-02 0.76 ± 0.43 3.68e-05 ± 1.18e-04 0.62 ± 0.48  10.46 ± 13.20       20.0          20     0.0
       500    PPA   1e-02 4.76e-02 ± 2.55e-02 0.45 ± 0.51 3.78e-06 ± 3.97e-06 0.57 ± 0.49  30.32 ± 35.36       20.0          20     0.0
       500    PPA   1e-01 5.00e-02 ± 2.56e-02 0.01 ± 0.01 6.57e-07 ± 7.28e-07 0.52 ± 0.49  71.51 ± 69.88       20.0          20     0.0
       500    PPA   1e+00 5.31e-02 ± 2.59e-02 0.00 ± 0.00 8.47e-08 ± 7.03e-08 0.48 ± 0.48 100.31 ± 27.32       20.0          20     0.0
       500    iPG       - 3.25e-02 ± 1.84e-02 1.00 ± 0.00 9.26e-10 ± 6.61e-10 0.85 ± 0.36  38.41 ± 64.76       20.0          20     0.0
"""


def parse_pm(token):
    token = token.strip()
    if token == "-":
        return np.nan, np.nan
    if "±" in token:
        mean, std = token.split("±")
        return float(mean.strip()), float(std.strip())
    return float(token), np.nan


rows = []
pattern = re.compile(
    r"^\s*(\d+)\s+(\S+)\s+(\S+)\s+"
    r"(.+?\s±\s.+?|-)"
    r"\s+(.+?\s±\s.+?|-)"
    r"\s+(.+?\s±\s.+?|-)"
    r"\s+(.+?\s±\s.+?|-)"
    r"\s+(.+?\s±\s.+?|-)"
    r"\s+(\S+)\s+(\S+)\s+(\S+)\s*$"
)

for line in raw.strip().splitlines():
    if line.startswith("n_samples"):
        continue
    m = pattern.match(line.rstrip())
    if not m:
        print("Could not parse:", line)
        continue

    n_samples, method, epsilon, loss, sparsity, feas, acc, time, n_success, n_attempts, n_fail = (
        m.groups()
    )
    loss_mean, loss_std = parse_pm(loss)
    sparsity_mean, sparsity_std = parse_pm(sparsity)
    feas_mean, feas_std = parse_pm(feas)
    acc_mean, acc_std = parse_pm(acc)
    time_mean, time_std = parse_pm(time)

    rows.append(
        {
            "n_samples": int(n_samples),
            "method": method,
            "epsilon": epsilon,
            "loss_mean": loss_mean,
            "loss_std": loss_std,
            "sparsity_mean": sparsity_mean,
            "sparsity_std": sparsity_std,
            "feas_mean": feas_mean,
            "feas_std": feas_std,
            "acc_mean": acc_mean,
            "acc_std": acc_std,
            "time_mean": time_mean,
            "time_std": time_std,
        }
    )

df = pd.DataFrame(rows)

# rename PGD -> EPGD
df["method"] = df["method"].replace({"PGD": "EPGD"})

df = df.sort_values(["method", "epsilon", "n_samples"])

linestyle_map = {
    "BAPG": "-",
    "CG": "--",
    "EPGD": "-.",
    "PPA": ":",
    "iPG": (0, (3, 1, 1, 1)),
}

marker_map = {
    "-": "o",
    "1e-03": "o",
    "1e-02": "s",
    "1e-01": "^",
    "1e+00": "D",
}

plt.figure(figsize=(12, 8))

for (method, epsilon), g in df.groupby(["method", "epsilon"]):
    g = g.sort_values("n_samples")
    label = method if epsilon == "-" else f"{method} ({epsilon})"
    plt.semilogy(
        g["n_samples"],
        g["loss_mean"],
        linestyle=linestyle_map.get(method, "-"),
        marker=marker_map.get(epsilon, "o"),
        linewidth=5,
        markersize=8,
        label=label,
    )

plt.xlabel("Number of Samples", fontsize=20)
plt.ylabel("Average Loss", fontsize=20)
# plt.title("Loss vs Number of Samples", fontsize=18)
plt.xticks(sorted(df["n_samples"].unique()), fontsize=20)
plt.yticks(fontsize=20)
plt.legend(fontsize=14, ncol=5, loc="upper center", bbox_to_anchor=(0.5, 1.18))
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(outdir, "loss_vs_nsamples.pdf"), format="pdf", bbox_inches="tight")
plt.show()

In [ ]:
plt.figure(figsize=(12, 8))

for (method, epsilon), g in df.groupby(["method", "epsilon"]):
    g = g.sort_values("n_samples")
    label = method if epsilon == "-" else f"{method} ({epsilon})"
    plt.semilogy(
        g["n_samples"],
        g["sparsity_mean"],
        linestyle=linestyle_map.get(method, "-"),
        marker=marker_map.get(epsilon, "o"),
        linewidth=5,
        markersize=8,
        label=label,
    )

plt.xlabel("Number of Samples", fontsize=20)
plt.ylabel("Average Sparsity", fontsize=20)
# plt.title("Sparsity vs Number of Samples", fontsize=20)
plt.xticks(sorted(df["n_samples"].unique()), fontsize=20)
plt.yticks(fontsize=20)
plt.legend(fontsize=14, ncol=5, loc="upper center", bbox_to_anchor=(0.5, 1.18))
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(outdir, "sparsity_vs_nsamples.pdf"), format="pdf", bbox_inches="tight")
plt.show()

In [ ]:
df.loc[(df["method"] == "CG") & (df["feas_mean"] < 1e-15), "feas_mean"] = 1e-17
plt.figure(figsize=(12, 8))

for (method, epsilon), g in df.groupby(["method", "epsilon"]):
    g = g.sort_values("n_samples")
    label = method if epsilon == "-" else f"{method} ({epsilon})"
    plt.semilogy(
        g["n_samples"],
        g["feas_mean"],
        linestyle=linestyle_map.get(method, "-"),
        marker=marker_map.get(epsilon, "o"),
        linewidth=5,
        markersize=8,
        label=label,
    )

plt.xlabel("Number of Samples", fontsize=20)
plt.ylabel("Average Feasibility", fontsize=20)
# plt.title("Feasibility vs Number of Samples", fontsize=20)
plt.xticks(sorted(df["n_samples"].unique()), fontsize=20)
plt.yticks(fontsize=20)
plt.legend(fontsize=14, ncol=5, loc="upper center", bbox_to_anchor=(0.5, 1.18))
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(outdir, "feasibility_vs_nsamples.pdf"), format="pdf", bbox_inches="tight")
plt.show()

In [ ]:
plt.figure(figsize=(12, 8))

for (method, epsilon), g in df.groupby(["method", "epsilon"]):
    g = g.sort_values("n_samples")
    label = method if epsilon == "-" else f"{method} ({epsilon})"
    plt.plot(
        g["n_samples"],
        g["acc_mean"],
        linestyle=linestyle_map.get(method, "-"),
        marker=marker_map.get(epsilon, "o"),
        linewidth=5,
        markersize=8,
        label=label,
    )

plt.xlabel("Number of Samples", fontsize=20)
plt.ylabel("Average Accuracy", fontsize=20)
# plt.title("Accuracy vs Number of Samples", fontsize=20)
plt.xticks(sorted(df["n_samples"].unique()), fontsize=20)
plt.yticks(fontsize=20)
plt.legend(fontsize=14, ncol=5, loc="upper center", bbox_to_anchor=(0.5, 1.18))
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(outdir, "accuracy_vs_nsamples.pdf"), format="pdf", bbox_inches="tight")
plt.show()

In [ ]:
plt.figure(figsize=(12, 8))

for (method, epsilon), g in df.groupby(["method", "epsilon"]):
    g = g.sort_values("n_samples")
    label = method if epsilon == "-" else f"{method} ({epsilon})"
    plt.plot(
        g["n_samples"],
        g["time_mean"],
        linestyle=linestyle_map.get(method, "-"),
        marker=marker_map.get(epsilon, "o"),
        linewidth=5,
        markersize=8,
        label=label,
    )

plt.xlabel("Number of Samples", fontsize=20)
plt.ylabel("Average Time", fontsize=20)
# plt.title("Time vs Number of Samples", fontsize=20)
plt.xticks(sorted(df["n_samples"].unique()), fontsize=20)
plt.yticks(fontsize=20)
plt.legend(fontsize=14, ncol=5, loc="upper center", bbox_to_anchor=(0.5, 1.18))
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(outdir, "time_vs_nsamples.pdf"), format="pdf", bbox_inches="tight")
plt.show()